## 0. This checkout, not whatever is installed

`zou_lab_control_v2` is the one entry: importing it puts this checkout's eight layers
on the path, ahead of anything else.  It has to come **first** -- if a `zlc_*` module
was already imported from somewhere else, it refuses out loud rather than leaving two
copies in one kernel.  (If that happens: restart the kernel and run this cell first.)


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
while not (_here / 'zou_lab_control_v2').is_dir() and _here != _here.parent:
    _here = _here.parent
sys.path.insert(0, str(_here))

import zou_lab_control_v2

print('code in use:', zou_lab_control_v2.ROOT)


# zlc_atom: observation and orchestration

The two paths below share the same virtual devices and runtime plane. A camera measurement only observes frames; the experimenter owns `sequencer.load/fire` in the manual path, while `CalibrationTask` owns the complete pulse-to-report workflow in the automated path.

In [1]:
from pathlib import Path

import numpy as np
from zlc_runtime import SignalDataPlane

import zlc_atom
from zlc_atom.install import create_installation
from zlc_atom.nodes._framework.pulse_source import resolve_pulse
from zlc_atom.nodes.camera_measurement import CameraMeasurementNode
from zlc_atom.nodes.calibration import CalibrationTask
from zlc_atom.nodes.occupancy import OccupancyProcessor
from zlc_atom.nodes.calibration.calibration import FrameContract, calibrate

ROOT = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / 'pulses').is_dir())
package_identity = {'package': zlc_atom.__name__, 'version': zlc_atom.__version__}
package_identity

{'package': 'zlc_atom', 'version': '0.1.0'}

## 1. Manual path: the user owns the sequencer

The measurement is armed and collects exact frames. The pulse is resolved and fired by the caller, outside `CameraMeasurementNode`.

In [2]:
installation = create_installation('virtual')
plane = SignalDataPlane()
camera = installation.device('camera')
sequencer = installation.device('sequencer')

manual_pulse = resolve_pulse('calibration', search_paths=(ROOT / 'pulses',))
manual_measurement = CameraMeasurementNode(camera=camera, signal_plane=plane, producer='manual_measurement')
manual_capture = manual_measurement.prepare(repeat=1, frames_per_cycle=3)

# Explicit experimenter-owned excitation: measurement never calls these methods.
sequencer.load(manual_pulse.program)
sequencer.fire()
manual_result = manual_capture.collect()
manual_signal = manual_result.publication.value(manual_measurement.signal_key('frames'))
manual_observation = {
    'frames': len(manual_result.frames),
    'values_shape': manual_signal.values.shape,
    'dtype': manual_result.frames[0].image.dtype.str,
}
manual_observation

{'frames': 3, 'values_shape': (1, 1, 3, 32, 48), 'dtype': '<u2'}

In [3]:
manual_monitor_node = CameraMeasurementNode(camera=camera, signal_plane=plane, producer='manual_monitor')
manual_monitor = manual_monitor_node.monitor(buffer_frames=1)
sequencer.load(manual_pulse.program)
sequencer.fire()
monitor_record = manual_monitor.poll()
monitor_front = plane.freeze()
manual_monitor_observation = (
    monitor_record.image.shape,
    monitor_record.image.dtype.str,
    manual_monitor_node.signal_key('frames') in monitor_front.signals,
)
manual_monitor.close()
manual_monitor_observation

((32, 48), '<u2', True)

## 2. Automated path: the task owns the whole experiment

The task resolves one default `pulses/calibration.py` long-short-long bracket, loads/fires the sequencer for each repeat, calibrates from the long reference consensus and short readout, and publishes calibration plus report.

In [4]:
task_result = CalibrationTask(
    camera=camera,
    sequencer=sequencer,
    signal_plane=plane,
    pulse_search_paths=(ROOT / 'pulses',),
    expected_centers_xy=installation.world.geometry.site_centers_xy,
).run()

occupancy_node = OccupancyProcessor(task_result.calibration, signal_plane=plane)
occupancy_result = occupancy_node.process(task_result.short.frames)
auto_observation = {
    'reference_frames': len(task_result.reference.frames),
    'short_frames': len(task_result.short.frames),
    'report_published': task_result.publication is not None,
    'counts_shape': occupancy_result.counts.shape,
    'rate_shape': occupancy_result.rate.shape,
    'rate_mean': float(np.mean(occupancy_result.rate)),
}
auto_observation

{'reference_frames': 60,
 'short_frames': 30,
 'report_published': True,
 'counts_shape': (30, 6),
 'rate_shape': (30,),
 'rate_mean': 0.48333333333333334}

In [5]:
# The frozen mathematical oracle remains independent of the virtual device path.
with np.load(ROOT / 'tests/fixtures/main_readout_oracle.npz', allow_pickle=False) as oracle:
    oracle_result = calibrate(
        oracle['input_reference_frames'],
        oracle['input_short_frames'],
        frame_contract=FrameContract((34, 40), exposure_seconds=0.005),
        grid_shape=(2, 3),
        expected_centers_xy=oracle['centers_row_major'],
    )
    oracle_errors = int(np.count_nonzero(oracle_result.report['predictions'] != oracle['input_latent_occupancy']))
oracle_errors

29

In [6]:
plane.close()
installation.close()